# RAS review experiments

Select a **GPU runtime**, then **Run all**. This runs the focused quality comparisons and the controlled native traversal sweep. A full run can take substantial CPU time after GPU embedding finishes; progress is printed by seed, concept, predicate set and query.

The ZIP produced at the end contains per-query results, replay arrays, calibrated programs, native assets, environment/configuration records, and checksums. Send **ras_review_results.zip** back for interpretation.

New comparisons:
1. PQ64 with FP32 versus int4 stored heads, using one shared codebook and separately fitted calibration.
2. Materialized FP32 concept-logit tables (both full-precision and compiled-head scores); actual table composition is timed inside native HNSW.
3. Over-fetch and live filtering using one custom traversal function, with independent search-effort sweeps and randomized repeated timing.

Recall is conditional candidate-pool semantic recall in the quality experiment and compiled-eligibility traversal recall in the HNSW experiment. Larger vocabulary memory tables are arithmetic, not results from newly learned concepts. Existing paper results are not overwritten.


In [ ]:
#@title 1. Run settings
USE_DRIVE = True #@param {type:"boolean"}
SYNTHETIC_CHECK_ONLY = False #@param {type:"boolean"}
HNSW_QUERIES = 1000 #@param {type:"integer"}
TIMING_REPEATS = 3 #@param {type:"integer"}
OVERFETCH_MULTIPLIERS = ".75,1,1.5,2,3,4,6,8" #@param {type:"string"}
EF_MULTIPLIERS = "1,2" #@param {type:"string"}
SEEDS = "7,17,27" #@param {type:"string"}


In [ ]:
#@title 2. Set up a dedicated checkout
import os, pathlib, subprocess, sys, shutil, json
BRANCH = "codex/ras-review-experiments"
REPO = "https://github.com/hanialshater/ras.git"
ROOT = pathlib.Path("/content/ras-review-experiments")
if not ROOT.exists():
    subprocess.run(["git", "clone", "--branch", BRANCH, "--single-branch", REPO, str(ROOT)], check=True)
else:
    origin = subprocess.check_output(["git", "-C", str(ROOT), "remote", "get-url", "origin"], text=True).strip()
    if origin != REPO:
        raise RuntimeError("Checkout directory belongs to another repository.")
    if subprocess.check_output(["git", "-C", str(ROOT), "status", "--porcelain", "--untracked-files=no"], text=True).strip():
        raise RuntimeError("Checkout has tracked local edits. Use a fresh runtime or preserve them before updating.")
    subprocess.run(["git", "-C", str(ROOT), "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", str(ROOT), "merge", "--ff-only", "origin/" + BRANCH], check=True)
os.chdir(ROOT)
COMMIT = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".[dev,benchmark]"], check=True)
os.environ["PATH"] = str(pathlib.Path.home() / ".cargo" / "bin") + os.pathsep + os.environ["PATH"]
if shutil.which("cargo") is None:
    subprocess.run(["bash", "-lc", "curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y --profile minimal"], check=True)
os.environ["PYTHONPATH"] = str(ROOT / "src") + os.pathsep + str(ROOT)
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
print("Commit:", COMMIT)
print(subprocess.check_output(["rustc", "--version"], text=True))


In [ ]:
#@title 3. Check regressions before spending time on models
subprocess.run([sys.executable, "-m", "pytest", "-q",
    "tests/test_binary.py", "tests/test_semantic_sidecar.py", "tests/test_accounting.py",
    "tests/test_review_regressions.py", "tests/test_smoke.py"], check=True)
subprocess.run(["cargo", "test", "--release", "--manifest-path",
    "rust/semantic_engine/Cargo.toml", "--bin", "semantic_hnsw_reviewer"], check=True)


In [ ]:
#@title 4. Save checkpoints and run experiments
import hashlib, time
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    BASE = pathlib.Path("/content/drive/MyDrive/ras_review")
else:
    BASE = pathlib.Path("/content/ras_review")
settings = dict(commit=COMMIT, synthetic=SYNTHETIC_CHECK_ONLY, queries=HNSW_QUERIES,
                repeats=TIMING_REPEATS, overfetch=OVERFETCH_MULTIPLIERS,
                ef=EF_MULTIPLIERS, seeds=SEEDS)
RUN_ID = hashlib.sha256(json.dumps(settings, sort_keys=True).encode()).hexdigest()[:12]
OUT = BASE / RUN_ID
OUT.mkdir(parents=True, exist_ok=True)
# Model/dataset cache can be reused across run IDs; it is excluded from results.
os.environ["HF_HOME"] = str(BASE / "hf_cache")
command = [sys.executable, "-u", "-m", "experiments.review_followup",
    "--output-dir", str(OUT), "--seeds", SEEDS,
    "--hnsw-queries", str(HNSW_QUERIES), "--timing-repeats", str(TIMING_REPEATS),
    "--overfetch-multipliers", OVERFETCH_MULTIPLIERS, "--ef-multipliers", EF_MULTIPLIERS]
if SYNTHETIC_CHECK_ONLY:
    command += ["--synthetic", "--seeds", "7"]
print("Results:", OUT, flush=True)
started = time.time()
subprocess.run(command, check=True)
print(f"Finished in {(time.time()-started)/60:.1f} minutes")


In [ ]:
#@title 5. Inspect quality and matched-recall results
import pandas as pd
summary = pd.read_csv(OUT / "summary.csv")
display(summary[(summary.retention == .2) & summary.metric.isin(["recall", "purity"])])
matched_path = OUT / "hnsw" / "matched_recall.csv"
if matched_path.exists():
    matched = pd.read_csv(matched_path)
    display(matched)
    print("Matched conditions:", int(matched.matched_within_tolerance.sum()), "/", len(matched))
    display(pd.read_csv(OUT / "hnsw" / "summary.csv").query("method == 'materialized_logits_hnsw'"))
print("Memory figures for large vocabularies are arithmetic only:")
display(pd.read_csv(OUT / "deployment_memory.csv").query("n_items == 5000000 and n_concepts in [8, 100000]"))


In [ ]:
#@title 6. Download the result bundle
from google.colab import files
BUNDLE = OUT / "ras_review_results.zip"
assert BUNDLE.exists(), "Run the experiment cell successfully first."
print(f"Download: {BUNDLE} ({BUNDLE.stat().st_size / 1e6:.1f} MB)")
files.download(str(BUNDLE))
